# Phase 2 Baseline + Hybrid (Colab Ready)

This notebook runs 4 evaluation tracks on `test.jsonl`:
1. `Llama-ZeroShot` (Groq)
2. `Llama-Prompted` (Groq)
3. `Mistral-Prompted` (HF Inference API)
4. `Hybrid` = Llama for strict structured fields + Mistral for description text

Outputs are saved to `/content/phase2_hybrid_outputs`.


In [ ]:
!pip install -q huggingface_hub scikit-learn rouge-score nltk pandas


In [ ]:
import os
import json
import re
import time
import glob
from pathlib import Path

import pandas as pd
import nltk
from huggingface_hub import InferenceClient
from sklearn.metrics import accuracy_score, f1_score, classification_report
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
print('Imports ready')


Imports ready


In [ ]:
# Paths (same style as your previous Colab notebook)
candidates = ['/content/test.jsonl', '/test.jsonl']
test_matches = [p for p in candidates if Path(p).exists()]
if not test_matches:
    test_matches = glob.glob('/kaggle/input/**/test.jsonl', recursive=True)

if not test_matches:
    raise FileNotFoundError('test.jsonl not found in /content or kaggle input')

TEST_FILE = test_matches[0]
DATA_DIR = str(Path(TEST_FILE).parent)
TRAIN_FILE = str(Path(DATA_DIR) / 'train.jsonl')
VAL_FILE = str(Path(DATA_DIR) / 'val.jsonl')
MASTER_FILE = str(Path(DATA_DIR) / 'master_with_splits.json')

if not Path(TRAIN_FILE).exists() and Path('/content/train.jsonl').exists():
    TRAIN_FILE = '/content/train.jsonl'
if not Path(VAL_FILE).exists() and Path('/content/val.jsonl').exists():
    VAL_FILE = '/content/val.jsonl'
if not Path(MASTER_FILE).exists() and Path('/content/master_with_splits.json').exists():
    MASTER_FILE = '/content/master_with_splits.json'
if not Path(MASTER_FILE).exists():
    MASTER_FILE = None

OUTPUT_DIR = '/content/phase2_hybrid_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('TEST_FILE  :', TEST_FILE)
print('TRAIN_FILE :', TRAIN_FILE, '| exists:', Path(TRAIN_FILE).exists())
print('VAL_FILE   :', VAL_FILE,   '| exists:', Path(VAL_FILE).exists())
print('MASTER_FILE:', MASTER_FILE)
print('OUTPUT_DIR :', OUTPUT_DIR)


TEST_FILE  : /content/test.jsonl
TRAIN_FILE : /content/train.jsonl | exists: True
VAL_FILE   : /content/val.jsonl | exists: True
MASTER_FILE: None
OUTPUT_DIR : /content/phase2_hybrid_outputs


In [ ]:
# HF token only (Colab/Kaggle/env fallback)
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
    except Exception:
        tok = os.environ.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise EnvironmentError('HF_TOKEN not set')
if not str(HF_TOKEN).startswith('hf_'):
    raise ValueError("HF_TOKEN is invalid. It must be a Hugging Face token starting with 'hf_'.")

print('HF_TOKEN loaded:', True, '| prefix:', HF_TOKEN[:6])


HF_TOKEN loaded  : True
GROQ_KEY loaded  : True


In [ ]:
# Model + client config (HF-only)
QWEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct:featherless-ai'
MISTRAL_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2:featherless-ai'

MAX_RETRIES = 3
RETRY_BASE_SEC = 1.5
DELAY_SEC = 1.0

hf_client = InferenceClient(token=os.environ['HF_TOKEN'], timeout=120)

print('QWEN_MODEL   :', QWEN_MODEL)
print('MISTRAL_MODEL:', MISTRAL_MODEL)


LLAMA_MODEL  : llama-3.3-70b-versatile
MISTRAL_MODEL: mistralai/Mistral-7B-Instruct-v0.2:featherless-ai


In [ ]:
# Data loading + dynamic label sets

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_master_records(master_path, fallback_rows):
    if not master_path:
        return fallback_rows
    with open(master_path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        if 'records' in obj and isinstance(obj['records'], list):
            return obj['records']
        if 'data' in obj and isinstance(obj['data'], list):
            return obj['data']
    return fallback_rows


test_records = load_jsonl(TEST_FILE)
master_records = load_master_records(MASTER_FILE, test_records)

VALID_CATEGORIES = sorted({str(r.get('category', '')).strip() for r in master_records if r.get('category')})
VALID_SEVERITIES = sorted({str(r.get('severity', '')).strip().lower() for r in master_records if r.get('severity')})
VALID_DEPARTMENTS = sorted({
    str(r.get('routing', {}).get('primary_department', '')).strip()
    for r in master_records if r.get('routing', {}).get('primary_department')
})

print(f'Loaded {len(test_records)} test records')
print('VALID_CATEGORIES :', VALID_CATEGORIES)
print('VALID_SEVERITIES :', VALID_SEVERITIES)
print('VALID_DEPARTMENTS:', VALID_DEPARTMENTS)


Loaded 32 test records
VALID_CATEGORIES : ['Broken Hospital Bed', 'Crowded Hospital Waiting Room', 'Dirty Hospital Bathroom', 'Empty / Unstaffed Nursing Station', 'Overflowing Hospital Trash (Outside)', 'Rats / Rodent Infestation', 'Torn Hospital Privacy Curtain', 'Unappetizing Hospital Food Tray', 'Unhygienic / Contaminated Hospital Food', 'Water Puddle on Hospital Floor']
VALID_SEVERITIES : ['critical', 'high', 'low', 'medium']
VALID_DEPARTMENTS: ['Administration', 'Dietary', 'Facilities Management', 'Housekeeping', 'Maintenance', 'Nursing', 'Pest Control', 'Waste Management']


In [ ]:
# Prompt builders

def build_zeroshot_prompt(caption, voice_text):
    return f"""A patient filed a hospital complaint.
Image caption: {caption}
Voice complaint: {voice_text}

Return JSON with keys:
category, severity, department, complaint_description"""


def build_prompted_prompt(caption, voice_text):
    cat_str = "\\n".join(f"- {c}" for c in VALID_CATEGORIES)
    dep_str = ", ".join(VALID_DEPARTMENTS)

    return f"""You are a hospital complaint triage system.
Return ONLY valid JSON (no markdown) with keys:
category, severity, department, complaint_description

Use exactly one category from:
{cat_str}

Use severity from:
- low
- medium
- high
- critical

Use department from:
{dep_str}

Examples:
Input: bathroom mold + patient says bathroom filthy
Output: {{"category":"Dirty Hospital Bathroom","severity":"high","department":"Housekeeping","complaint_description":"Patient reported severe unhygienic bathroom conditions with visible mold and foul odor."}}

Input: rat near kitchen + patient says rat seen near food area
Output: {{"category":"Rats / Rodent Infestation","severity":"critical","department":"Pest Control","complaint_description":"Patient observed rodent presence near food handling area, creating major hygiene risk."}}

Now process:
Image caption: {caption}
Voice complaint: {voice_text}
"""


def build_mistral_description_prompt(caption, voice_text, category, severity, department):
    return f"""You write only complaint description text for a hospital triage JSON.
Do not output JSON. Output only 1-2 sentences.
Stay grounded in given inputs and do not invent facts.

Image caption: {caption}
Voice complaint: {voice_text}
Chosen category: {category}
Chosen severity: {severity}
Chosen department: {department}

Write the complaint description now:"""


In [ ]:
# Parsing helpers

def parse_json_output(raw_text):
    if not raw_text:
        return {}, False

    cleaned = re.sub(r'```(?:json)?', '', str(raw_text), flags=re.IGNORECASE).replace('```', '').strip()
    s = cleaned.find('{')
    e = cleaned.rfind('}')
    if s != -1 and e != -1 and e > s:
        block = cleaned[s:e+1]
        try:
            return json.loads(block), True
        except Exception:
            pass

    try:
        return json.loads(cleaned), True
    except Exception:
        return {}, False


def canonicalize(value, allowed, lower=False):
    if value is None:
        return ''
    raw = str(value).strip()
    if lower:
        raw_cmp = raw.lower()
        lut = {x.lower(): x for x in allowed}
    else:
        raw_cmp = raw
        lut = {x: x for x in allowed}
    return lut.get(raw_cmp, raw.lower() if lower else raw)


def extract_fields(parsed):
    cat = canonicalize(parsed.get('category', ''), VALID_CATEGORIES, lower=False)
    sev = canonicalize(parsed.get('severity', ''), VALID_SEVERITIES, lower=True)
    dep = canonicalize(parsed.get('department', ''), VALID_DEPARTMENTS, lower=False)
    desc = str(parsed.get('complaint_description', '')).strip()
    return cat, sev, dep, desc


In [ ]:
# API callers (HF-only)

def _call_hf_chat(model_name, prompt, record_id, track_name, system_text='Follow user instruction exactly.'):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if hasattr(hf_client, 'chat') and hasattr(hf_client.chat, 'completions'):
                resp = hf_client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {'role': 'system', 'content': system_text},
                        {'role': 'user', 'content': prompt},
                    ],
                    temperature=0,
                    max_tokens=512,
                )
            else:
                resp = hf_client.chat_completion(
                    model=model_name,
                    messages=[
                        {'role': 'system', 'content': system_text},
                        {'role': 'user', 'content': prompt},
                    ],
                    temperature=0,
                    max_tokens=512,
                )

            content = resp.choices[0].message.content
            if isinstance(content, str):
                return content.strip()
            if isinstance(content, list):
                return ''.join((x.get('text', '') if isinstance(x, dict) else str(x)) for x in content).strip()
            return str(content).strip()
        except Exception as e:
            last_err = str(e)
            retryable = any(k in last_err.lower() for k in ['429', 'rate', 'timeout', '503', '502'])
            if retryable and attempt < MAX_RETRIES:
                time.sleep(RETRY_BASE_SEC * (2 ** (attempt - 1)))
                continue
            break

    print(f'  ERROR on {record_id} [{track_name}-HF]: {last_err}')
    return None


def call_hf_qwen(prompt, record_id, track_name):
    return _call_hf_chat(
        model_name=QWEN_MODEL,
        prompt=prompt,
        record_id=record_id,
        track_name=track_name,
        system_text='Return only valid JSON when asked for JSON.'
    )


def call_hf_mistral(prompt, record_id, track_name):
    return _call_hf_chat(
        model_name=MISTRAL_MODEL,
        prompt=prompt,
        record_id=record_id,
        track_name=track_name,
        system_text='Follow user instruction exactly.'
    )


In [ ]:
# Metrics + runners

def compute_metrics(results, track_name, model_name):
    gt_c = [r['gt_category'] for r in results]
    gt_s = [r['gt_severity'] for r in results]
    gt_d = [r['gt_department'] for r in results]

    pr_c = [r['pred_category'] for r in results]
    pr_s = [r['pred_severity'] for r in results]
    pr_d = [r['pred_department'] for r in results]

    json_valid_count = sum(1 for r in results if r['json_valid'])

    cat_acc = accuracy_score(gt_c, pr_c)
    sev_acc = accuracy_score(gt_s, pr_s)
    dep_acc = accuracy_score(gt_d, pr_d)

    cat_f1 = f1_score(gt_c, pr_c, average='macro', zero_division=0)
    sev_f1 = f1_score(gt_s, pr_s, average='macro', zero_division=0)
    dep_f1 = f1_score(gt_d, pr_d, average='macro', zero_division=0)

    invalid_cat = sum(1 for x in pr_c if x not in VALID_CATEGORIES)
    invalid_sev = sum(1 for x in pr_s if x not in VALID_SEVERITIES)
    invalid_dep = sum(1 for x in pr_d if x not in VALID_DEPARTMENTS)

    smoother = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    bleu, r1, r2, rl = [], [], [], []
    for r in results:
        ref = r['gt_description'].split()
        hyp = r['pred_description'].split()
        if ref and hyp:
            bleu.append(sentence_bleu([ref], hyp, smoothing_function=smoother))
            rs = scorer.score(r['gt_description'], r['pred_description'])
            r1.append(rs['rouge1'].fmeasure)
            r2.append(rs['rouge2'].fmeasure)
            rl.append(rs['rougeL'].fmeasure)

    def avg(vals):
        return round(sum(vals) / len(vals), 4) if vals else 0.0

    print(f"\nPer-class CATEGORY report [{track_name}]")
    print(classification_report(gt_c, pr_c, zero_division=0))

    return {
        'baseline': track_name,
        'model': model_name,
        'total_records': len(results),
        'json_validity_rate': round(json_valid_count / len(results), 4),
        'category_accuracy': round(cat_acc, 4),
        'category_macro_f1': round(cat_f1, 4),
        'severity_accuracy': round(sev_acc, 4),
        'severity_macro_f1': round(sev_f1, 4),
        'department_accuracy': round(dep_acc, 4),
        'department_macro_f1': round(dep_f1, 4),
        'bleu_score': avg(bleu),
        'rouge1': avg(r1),
        'rouge2': avg(r2),
        'rougeL': avg(rl),
        'invalid_category_preds': invalid_cat,
        'invalid_severity_preds': invalid_sev,
        'invalid_department_preds': invalid_dep,
    }


def run_track(rows, track_name, json_prompt_builder, json_caller, hybrid_merge=False):
    print("\n" + "=" * 66)
    print(f"Running: {track_name} ({len(rows)} records)")
    print("=" * 66)

    out = []
    ckpt = os.path.join(OUTPUT_DIR, f"checkpoint_{track_name.replace(' ', '_')}.csv")

    for i, rec in enumerate(rows, 1):
        image_id = rec.get('image_id', f'row_{i}')
        caption = rec.get('input', {}).get('image_caption', rec.get('refined_caption', ''))
        voice = rec.get('voice_text', '')

        gt_cat = str(rec.get('category', '')).strip()
        gt_sev = str(rec.get('severity', '')).strip().lower()
        gt_dep = str(rec.get('routing', {}).get('primary_department', rec.get('department', ''))).strip()
        gt_desc = str(rec.get('complaint_description', '')).strip()

        print(f"  [{i:02d}/{len(rows)}] {image_id} ... ", end='', flush=True)

        base_prompt = json_prompt_builder(caption, voice)
        raw_base = json_caller(base_prompt, image_id, track_name)
        parsed_base, base_valid = parse_json_output(raw_base)
        b_cat, b_sev, b_dep, b_desc = extract_fields(parsed_base)

        pr_cat, pr_sev, pr_dep, pr_desc = b_cat, b_sev, b_dep, b_desc

        if hybrid_merge:
            # Best-field routing from your reported metrics:
            # Qwen better: category + severity
            # Mistral better: department + description
            raw_m = call_hf_mistral(build_prompted_prompt(caption, voice), image_id, track_name)
            parsed_m, m_valid = parse_json_output(raw_m)
            m_cat, m_sev, m_dep, m_desc = extract_fields(parsed_m)

            # category (prefer Qwen)
            pr_cat = b_cat if b_cat in VALID_CATEGORIES else (m_cat if m_cat in VALID_CATEGORIES else b_cat)
            # severity (prefer Qwen)
            pr_sev = b_sev if b_sev in VALID_SEVERITIES else (m_sev if m_sev in VALID_SEVERITIES else b_sev)
            # department (prefer Mistral)
            pr_dep = m_dep if m_dep in VALID_DEPARTMENTS else (b_dep if b_dep in VALID_DEPARTMENTS else m_dep)
            # description (prefer Mistral)
            pr_desc = m_desc if m_desc else b_desc

            is_valid = base_valid or m_valid
            raw_output = json.dumps({'qwen_raw': raw_base, 'mistral_raw': raw_m}, ensure_ascii=False)

            src_cat = 'qwen'
            src_sev = 'qwen'
            src_dep = 'mistral'
            src_desc = 'mistral'
        else:
            is_valid = base_valid
            raw_output = raw_base or ''
            src_cat = 'single_model'
            src_sev = 'single_model'
            src_dep = 'single_model'
            src_desc = 'single_model'

        cat_ok = (pr_cat == gt_cat)
        sev_ok = (pr_sev == gt_sev)
        dep_ok = (pr_dep == gt_dep)

        print(('cat✓' if cat_ok else 'cat✗'), '|', ('sev✓' if sev_ok else 'sev✗'), '|', ('json✓' if is_valid else 'json✗'))

        out.append({
            'baseline': track_name,
            'image_id': image_id,
            'gt_category': gt_cat,
            'gt_severity': gt_sev,
            'gt_department': gt_dep,
            'gt_description': gt_desc,
            'pred_category': pr_cat,
            'pred_severity': pr_sev,
            'pred_department': pr_dep,
            'pred_description': pr_desc,
            'json_valid': is_valid,
            'raw_output': raw_output,
            'cat_correct': cat_ok,
            'sev_correct': sev_ok,
            'dept_correct': dep_ok,
            'source_category': src_cat,
            'source_severity': src_sev,
            'source_department': src_dep,
            'source_description': src_desc,
        })

        pd.DataFrame(out).to_csv(ckpt, index=False)
        time.sleep(DELAY_SEC)

    return out


In [ ]:
# Smoke tests
print('Smoke test Qwen:')
smoke_q = call_hf_qwen(
    build_zeroshot_prompt('A broken hospital bed with unsafe rail', 'My bed is broken and unsafe'),
    'smoke_qwen',
    'Smoke',
)
print(smoke_q)

print('\nSmoke test Mistral:')
smoke_m = call_hf_mistral(
    build_prompted_prompt('A broken hospital bed with unsafe rail', 'My bed is broken and unsafe'),
    'smoke_mistral',
    'Smoke',
)
print(smoke_m)


Smoke test Llama:
```json
{
  "category": "Equipment/Facilities",
  "severity": "High",
  "department": "Maintenance/Patient Care",
  "complaint_description": "Broken hospital bed with unsafe rail"
}
```
\nSmoke test Mistral:
Output: {
"category": "Broken Hospital Bed",
"severity": "high",
"department": "Maintenance",
"complaint_description": "Patient reported a broken hospital bed with an unsafe rail, posing a risk to patient safety."
}


In [ ]:
# Run all tracks (HF-only)
results_qwen_zs = run_track(test_records, 'Qwen-ZeroShot', build_zeroshot_prompt, call_hf_qwen, hybrid_merge=False)
metrics_qwen_zs = compute_metrics(results_qwen_zs, 'Qwen-ZeroShot', QWEN_MODEL)

results_qwen_pt = run_track(test_records, 'Qwen-Prompted', build_prompted_prompt, call_hf_qwen, hybrid_merge=False)
metrics_qwen_pt = compute_metrics(results_qwen_pt, 'Qwen-Prompted', QWEN_MODEL)

results_mistral_pt = run_track(test_records, 'Mistral-Prompted', build_prompted_prompt, call_hf_mistral, hybrid_merge=False)
metrics_mistral_pt = compute_metrics(results_mistral_pt, 'Mistral-Prompted', MISTRAL_MODEL)

results_hybrid = run_track(
    test_records,
    'Hybrid-QwenCatSev-MistralDeptDesc',
    build_prompted_prompt,
    call_hf_qwen,
    hybrid_merge=True,
)
metrics_hybrid = compute_metrics(
    results_hybrid,
    'Hybrid-QwenCatSev-MistralDeptDesc',
    f'{QWEN_MODEL} + {MISTRAL_MODEL}'
)

all_results = results_qwen_zs + results_qwen_pt + results_mistral_pt + results_hybrid
all_metrics = [metrics_qwen_zs, metrics_qwen_pt, metrics_mistral_pt, metrics_hybrid]

results_csv = os.path.join(OUTPUT_DIR, 'baseline_results.csv')
metrics_json = os.path.join(OUTPUT_DIR, 'baseline_metrics.json')
errors_csv = os.path.join(OUTPUT_DIR, 'baseline_errors.csv')
summary_txt = os.path.join(OUTPUT_DIR, 'run_summary.txt')

pd.DataFrame(all_results).to_csv(results_csv, index=False)
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

errors = [r for r in all_results if (not r['cat_correct']) or (not r['sev_correct']) or (not r['json_valid'])]
pd.DataFrame(errors).to_csv(errors_csv, index=False)

summary_cols = [
    'baseline','model','json_validity_rate',
    'category_accuracy','category_macro_f1',
    'severity_accuracy','severity_macro_f1',
    'department_accuracy','department_macro_f1',
    'bleu_score','rouge1','rouge2','rougeL',
]
summary_df = pd.DataFrame(all_metrics)[summary_cols]

with open(summary_txt, 'w', encoding='utf-8') as f:
    f.write(summary_df.to_string(index=False))

print('\nSaved files:')
print(' -', results_csv)
print(' -', metrics_json)
print(' -', errors_csv)
print(' -', summary_txt)

print('\nSummary table:')
print(summary_df)



Running: Llama-ZeroShot (32 records)
  [01/32] broken_9 ... cat✗ | sev✓ | json✓
  [02/32] broken_1 ... cat✗ | sev✓ | json✓
  [03/32] broken_4 ... cat✗ | sev✓ | json✓
  [04/32] broken_21 ... cat✗ | sev✗ | json✓
  [05/32] crowded_6 ... cat✗ | sev✗ | json✓
  [06/32] crowded_1 ... cat✗ | sev✗ | json✓
  [07/32] crowded_9 ... cat✗ | sev✗ | json✓
  [08/32] crowded_19 ... cat✗ | sev✗ | json✓
  [09/32] dirty_sc_12 ... cat✗ | sev✓ | json✓
  [10/32] dirty_sc_10 ... cat✗ | sev✓ | json✓
  [11/32] dirty_5 ... cat✗ | sev✓ | json✓
  [12/32] nursing_9 ... cat✗ | sev✗ | json✓
  [13/32] nursing_22 ... cat✗ | sev✗ | json✓
  [14/32] nursing_7 ... cat✗ | sev✗ | json✓
  [15/32] trash_11 ... cat✗ | sev✗ | json✓
  [16/32] trash_19 ... cat✗ | sev✗ | json✓
  [17/32] trash_7 ... cat✗ | sev✗ | json✓
  [18/32] rats_sc_3 ... cat✗ | sev✗ | json✓
  [19/32] rats_8 ... cat✗ | sev✗ | json✓
  [20/32] rats_9 ... cat✗ | sev✗ | json✓
  [21/32] rats_sc_5 ... cat✗ | sev✗ | json✓
  [22/32] curtain_15 ... cat✗ | sev✗ | json✓
  

In [ ]:
import os, zipfile
from datetime import datetime
from google.colab import files

print("OUTPUT_DIR:", OUTPUT_DIR)
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp):
        print("-", fn)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f"/content/phase2_hybrid_outputs_{ts}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        fp = os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, arcname=fn)

print("Created:", zip_path)
files.download(zip_path)


OUTPUT_DIR: /content/phase2_hybrid_outputs
- baseline_errors.csv
- baseline_metrics.json
- baseline_results.csv
- checkpoint_Hybrid-LlamaCat-MistralSevDeptDesc.csv
- checkpoint_Hybrid-LlamaStruct-MistralDesc.csv
- checkpoint_Llama-Prompted.csv
- checkpoint_Llama-ZeroShot.csv
- checkpoint_Mistral-Prompted.csv
- run_summary.txt
Created: /content/phase2_hybrid_outputs_20260425_083519.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>